# Step 5 Training — Minimal Colab Version (upload/download, no Drive)

**First: set the GPU runtime.** `Runtime` menu → `Change runtime type` → `Hardware accelerator` → **T4 GPU** → Save.

This notebook expects ONE file: `colab_training_bundle.zip`, built locally by the commands in the accompanying instructions. It contains only the code and data training actually needs — nothing else.

## 1. Upload the bundle

In [ ]:
from google.colab import files
uploaded = files.upload()  # select colab_training_bundle.zip when prompted

In [ ]:
!mkdir -p /content/project
!unzip -q colab_training_bundle.zip -d /content/project
%cd /content/project
!find . -type f

Confirm you see `training/`, `models/`, `datasets/`, `retrieval/analogues_top20.parquet`, `evaluation/normalization_stats.json`. If anything's missing, the local zip step was incomplete — go back and re-check it before continuing.

## 2. Dependencies (pandas/pyarrow/torch are already in Colab — nothing extra needed for training itself; faiss-cpu is only needed for retrieval, which isn't run here)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- go back and set the GPU runtime")

## 3. Train — 5 seeds

This is the actual heavy step. `training/train_retrieval_lstm.py` auto-detects the GPU and uses it automatically. If the session disconnects partway, re-run with just the missing seed numbers, e.g. `!python -m training.train_retrieval_lstm 3 4` — it merges into the existing results rather than starting over.

In [ ]:
!python -m training.train_retrieval_lstm

## 4. Zip up the results and download them back to your machine

In [ ]:
!zip -r /content/retrieval_augmented_results.zip models/retrieval_augmented evaluation/retrieval_augmented predictions/retrieval_augmented
from google.colab import files
files.download('/content/retrieval_augmented_results.zip')

## 5. Back on your own machine

Extract `retrieval_augmented_results.zip` into your project root — it will drop `models/retrieval_augmented/seed_{0-4}/checkpoint.pt`, `evaluation/retrieval_augmented/*.json`, and `predictions/retrieval_augmented/val_predictions.parquet` into the right places automatically.

Then, entirely locally, no GPU needed:
```bash
python -m training.evaluate_retrieval_lstm
```
This reads the checkpoints you just brought back and computes the stratified metrics — same fast, CPU-only evaluation as everything else in this project.

**Expected numbers** (yours should match closely — training is deterministic given the fixed seeds, though GPU vs CPU floating-point can cause tiny differences):

| | Extreme RMSE | Recall | F2 |
|---|---|---|---|
| Canonical baseline | 1.285 | 0.339 | 0.322 |
| Retrieval-augmented | ~1.20 | ~0.40 | ~0.36 |